# hippofloop: Distill floop's consolidator into a local model

Fine-tunes **Qwen 2.5 3B Instruct** (4-bit QLoRA) on floop's decision logs.
Runs end-to-end on a Kaggle T4 GPU.

**Pipeline:** Load JSONL → Clean → Format SFT pairs → Train → Evaluate → Export GGUF

## 1. Setup

In [ ]:
%%capture
# Install unsloth (optimized for Kaggle/Colab T4)
!pip install unsloth[colab-new]
!pip install --no-deps trl peft accelerate bitsandbytes xformers

# Install hippofloop from repo
!pip install git+https://github.com/nvandessel/hippofloop.git

In [ ]:
import json
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("hippofloop.notebook")

## 2. Load and explore data

Data is expected as a Kaggle Dataset. The path is auto-detected.

In [ ]:
# Auto-detect Kaggle dataset path
_candidates = [
    Path("/kaggle/input/datasets/nvandessel/floop-decisions"),
    Path("/kaggle/input/floop-decisions"),
]
DATA_DIR = next((p for p in _candidates if p.exists()), _candidates[-1])
print(f"Data dir: {DATA_DIR}")

# Find all JSONL files in the dataset
jsonl_files = sorted(DATA_DIR.glob("*.jsonl"))
print(f"Found {len(jsonl_files)} JSONL files:")
for f in jsonl_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name} ({size_mb:.1f} MB)")

In [ ]:
from hippofloop.data.loader import JsonlLoader
from hippofloop.data.cleaner import DecisionCleaner
from hippofloop.data.formatter import SftFormatter

loader = JsonlLoader()
cleaner = DecisionCleaner()
formatter = SftFormatter()

# Load
entries = loader.load([str(f) for f in jsonl_files])
print(f"Loaded: {len(entries)} entries")

# Clean
cleaned, stats = cleaner.clean_with_stats(entries)
print(f"\nCleaning stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")

# Format
pairs = formatter.format(cleaned)
print(f"\nSFT pairs: {len(pairs)}")

# Per-task breakdown
from collections import Counter
task_counts = Counter(p.task for p in pairs)
print("\nBy task:")
for task, count in sorted(task_counts.items()):
    print(f"  {task}: {count}")

In [ ]:
# Inspect a sample SFT pair
sample = pairs[0]
print(f"Task: {sample.task}")
print(f"Source stage: {sample.source_stage}")
for msg in sample.messages:
    content_preview = msg['content'][:200] + '...' if len(msg['content']) > 200 else msg['content']
    print(f"\n[{msg['role']}]\n{content_preview}")

## 3. Split data

In [ ]:
SEED = 42

train_pairs, val_pairs, test_pairs = formatter.split(
    pairs, train_ratio=0.8, val_ratio=0.1, seed=SEED
)
print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}, Test: {len(test_pairs)}")

# Per-task distribution in train split
train_tasks = Counter(p.task for p in train_pairs)
print("\nTrain split by task:")
for task, count in sorted(train_tasks.items()):
    print(f"  {task}: {count}")

## 4. Load model and apply LoRA

In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 8192

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

# Print trainable params
model.print_trainable_parameters()

## 5. Prepare datasets

In [ ]:
from datasets import Dataset
from hippofloop.training.trainer import UnslothTrainer
from hippofloop.training.config import TrainingConfig

# Use UnslothTrainer.prepare_dataset for consistent formatting
config = TrainingConfig(
    base_model=BASE_MODEL,
    lora_rank=32, lora_alpha=64, lora_dropout=0.05,
    lora_target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    learning_rate=2e-4, lr_scheduler="cosine", warmup_ratio=0.03,
    epochs=3, batch_size=1, gradient_accumulation_steps=16,
    max_seq_length=MAX_SEQ_LENGTH, weight_decay=0.01,
    bf16=False, fp16=True,
    train_split=0.8, val_split=0.1, test_split=0.1, seed=SEED,
    quantization="Q4_K_M", output_path="hippofloop.gguf",
)

trainer_helper = UnslothTrainer(config)
train_dataset = Dataset.from_list(trainer_helper.prepare_dataset(train_pairs))
val_dataset = Dataset.from_list(trainer_helper.prepare_dataset(val_pairs))

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset: {len(val_dataset)} examples")

## 6. Train

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = "checkpoints/qwen25-3b-hippofloop"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    bf16=False,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    seed=SEED,
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

print(f"Training {len(train_dataset)} examples for {training_args.num_train_epochs} epochs...")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
sft_trainer.train()

In [ ]:
# Training loss curve
import matplotlib.pyplot as plt

train_losses = [(e["step"], e["loss"]) for e in sft_trainer.state.log_history if "loss" in e]
eval_losses = [(e["step"], e["eval_loss"]) for e in sft_trainer.state.log_history if "eval_loss" in e]

fig, ax = plt.subplots(figsize=(10, 5))
if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label="Train loss", alpha=0.7)
if eval_losses:
    steps, losses = zip(*eval_losses)
    ax.plot(steps, losses, "o-", label="Eval loss", markersize=8)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Training Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Best eval loss
best_loss = None
if eval_losses:
    best_step, best_loss = min(eval_losses, key=lambda x: x[1])
    print(f"Best eval loss: {best_loss:.4f} at step {best_step}")

## 7. Save best model

In [ ]:
BEST_MODEL_PATH = f"{OUTPUT_DIR}/best"
sft_trainer.save_model(BEST_MODEL_PATH)
tokenizer.save_pretrained(BEST_MODEL_PATH)
print(f"Saved best model to {BEST_MODEL_PATH}")

## 8. Evaluate on test set

In [ ]:
import random

from hippofloop.eval.evaluator import ModelEvaluator

# Set up inference mode
FastLanguageModel.for_inference(model)

def model_fn(system_msg: str, user_msg: str) -> str:
    """Run inference on the fine-tuned model."""
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=2048, temperature=0.1, do_sample=True
    )
    # Decode only the generated tokens
    generated = outputs[0][inputs.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

# Run eval on a representative random subset (full eval can be slow)
rng = random.Random(SEED)
eval_subset = rng.sample(test_pairs, min(50, len(test_pairs)))
print(f"Evaluating on {len(eval_subset)} test examples...")

evaluator = ModelEvaluator(model_fn)
results = evaluator.evaluate(eval_subset)
report = evaluator.summary_report(results)

print(f"\nOverall:")
print(f"  JSON valid rate: {report['json_valid_rate']:.1%}")
print(f"  Schema valid rate: {report['schema_valid_rate']:.1%}")
print(f"\nBy task:")
for task, task_report in report.get('by_task', {}).items():
    print(f"  {task} (n={task_report['count']}):")
    print(f"    JSON valid: {task_report['json_valid_rate']:.1%}")
    print(f"    Schema valid: {task_report['schema_valid_rate']:.1%}")

## 9. Export to GGUF

In [ ]:
GGUF_OUTPUT = "hippofloop-qwen25-3b-Q4_K_M.gguf"

model.save_pretrained_gguf(
    "gguf_export",
    tokenizer,
    quantization_method="q4_k_m",
)

# Rename to target filename
gguf_dir = Path("gguf_export")
gguf_files = list(gguf_dir.glob("*.gguf"))
if gguf_files:
    actual = max(gguf_files, key=lambda p: p.stat().st_mtime)
    actual.rename(GGUF_OUTPUT)
    size_mb = Path(GGUF_OUTPUT).stat().st_size / (1024 * 1024)
    print(f"Exported: {GGUF_OUTPUT} ({size_mb:.0f} MB)")
else:
    print("Warning: no GGUF file found after export")

## 10. Download

The GGUF file is saved in the notebook's working directory.
On Kaggle, go to **Output** tab to download it, or save as a Kaggle Dataset for reuse.

In [ ]:
# Summary
print("=" * 60)
print("hippofloop training complete")
print("=" * 60)
print(f"Base model: {BASE_MODEL}")
print(f"Training examples: {len(train_pairs)}")
print(f"Validation examples: {len(val_pairs)}")
print(f"Test examples: {len(test_pairs)}")
if best_loss is not None:
    print(f"Best eval loss: {best_loss:.4f}")
print(f"JSON valid rate: {report['json_valid_rate']:.1%}")
print(f"Schema valid rate: {report['schema_valid_rate']:.1%}")
print(f"GGUF output: {GGUF_OUTPUT}")